# Experiment 01 — HUGS Locality Probe

Google Colab setup for the first **Interactive Digital Humans / 4D Human Intelligence** diagnostic experiment.


## 0. Runtime

Choose **A100 GPU** when available.


In [ ]:
!nvidia-smi
import torch, platform
print("Colab Python:", platform.python_version())
print("Colab PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Clone repositories


In [ ]:
%cd /content
!rm -rf interactive-digital-humans ml-hugs
!git clone https://github.com/reusahn/interactive-digital-humans.git
!git clone --recursive https://github.com/apple-aiml-research/ml-hugs.git


## 2. Create pinned HUGS environment


In [ ]:
import os, subprocess
from pathlib import Path

def run_live(cmd, cwd=None, env=None):
    print("\n+", cmd, flush=True)
    p = subprocess.run(
        cmd, shell=True, cwd=cwd, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(p.stdout)
    if p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {cmd}")
    return p

MINIFORGE = Path("/content/miniforge")
CONDA = MINIFORGE / "bin" / "conda"
PREFIX = MINIFORGE / "envs" / "hugs"

if not CONDA.exists():
    run_live("wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /content/miniforge.sh")
    run_live("bash /content/miniforge.sh -b -p /content/miniforge")

if not (PREFIX / "bin" / "python").exists():
    run_live(f"{CONDA} create -n hugs python=3.8 pip -y -c conda-forge")

run_live(f"{CONDA} install -n hugs -y -c nvidia/label/cuda-11.7.0 cuda-toolkit=11.7.0")
run_live(f"{CONDA} install -n hugs -y -c conda-forge gcc_linux-64=11 gxx_linux-64=11")

PY = str(PREFIX / "bin" / "python")
PIP = f"{PY} -m pip"

run_live(f"{PIP} install --upgrade 'pip<25' 'setuptools<70' wheel ninja")
run_live(f"{PIP} install torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117")
run_live(f"{PIP} install numpy==1.23.5 fvcore iopath")
run_live(f"{PIP} install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py38_cu117_pyt1131/download.html")


## 2B. Build HUGS CUDA extensions


In [ ]:
build_env = os.environ.copy()
build_env["CUDA_HOME"] = str(PREFIX)
build_env["PATH"] = f"{PREFIX}/bin:" + build_env.get("PATH", "")
build_env["LD_LIBRARY_PATH"] = f"{PREFIX}/lib:{PREFIX}/lib64:" + build_env.get("LD_LIBRARY_PATH", "")
build_env["CC"] = f"{PREFIX}/bin/x86_64-conda-linux-gnu-cc"
build_env["CXX"] = f"{PREFIX}/bin/x86_64-conda-linux-gnu-c++"
build_env["CUDAHOSTCXX"] = build_env["CXX"]
build_env["TORCH_CUDA_ARCH_LIST"] = "8.0"
build_env["MAX_JOBS"] = "2"
build_env["FORCE_CUDA"] = "1"

import shlex

run_live(f"{PREFIX}/bin/nvcc --version", env=build_env)
run_live(f"{build_env['CC']} --version", env=build_env)

check_code = "import torch; print(torch.__version__); print(torch.version.cuda); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))"
run_live(f"{PY} -c {shlex.quote(check_code)}", env=build_env)

run_live(
    f"{PIP} install -v --no-build-isolation ./submodules/diff-gaussian-rasterization",
    cwd="/content/ml-hugs", env=build_env
)
run_live(
    f"{PIP} install -v --no-build-isolation ./submodules/simple-knn",
    cwd="/content/ml-hugs", env=build_env
)


## 2C. Install remaining dependencies


In [ ]:
run_live(f"{PIP} install -r requirements.txt", cwd="/content/ml-hugs", env=build_env)
run_live(f"{PIP} install git+https://github.com/mattloper/chumpy.git", cwd="/content/ml-hugs", env=build_env)

verify_code = "import torch, diff_gaussian_rasterization, simple_knn; print(torch.__version__); print(torch.version.cuda); print(torch.cuda.get_device_name(0)); print('CUDA extensions: OK')"
run_live(f"{PY} -c {shlex.quote(verify_code)}", cwd="/content/ml-hugs", env=build_env)


## 3. Download NeuMan data and pretrained HUGS models


In [ ]:
%cd /content/ml-hugs
!bash scripts/prepare_data_models.sh


## 4. Upload licensed SMPL neutral model

Download **SMPL v1.1.0** from the official SMPL site and upload the neutral `.pkl`.


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()
smpl_dir = Path("/content/ml-hugs/data/smpl")
smpl_dir.mkdir(parents=True, exist_ok=True)

for name in uploaded:
    src = Path(name)
    if name.lower().endswith(".pkl"):
        dst = smpl_dir / "SMPL_NEUTRAL.pkl"
    elif name == "smpl_uv.obj":
        dst = smpl_dir / "smpl_uv.obj"
    else:
        continue
    shutil.move(str(src), str(dst))
    print("Saved:", dst)

assert (smpl_dir / "SMPL_NEUTRAL.pkl").exists(), "Upload the neutral SMPL .pkl file."


## 5. Find a pretrained HUGS output directory


In [ ]:
from pathlib import Path
root = Path("/content/ml-hugs")
configs = list(root.rglob("config_train.yaml"))
print("Found", len(configs), "candidate pretrained outputs")
for i, p in enumerate(configs):
    print(i, p.parent)


In [ ]:
HUGS_OUTPUT_DIR = ""  # paste a human or human_scene directory printed above
p = Path(HUGS_OUTPUT_DIR)
assert HUGS_OUTPUT_DIR and (p / "config_train.yaml").exists(), "Set HUGS_OUTPUT_DIR first."
print("Using:", p)


## 6. Official HUGS evaluation sanity check


In [ ]:
run_live(
    f'{PY} scripts/evaluate.py -o "{HUGS_OUTPUT_DIR}"',
    cwd="/content/ml-hugs", env=build_env
)


## 7. Run locality probe: left wrist +10°


In [ ]:
cmd = (
    f'{PY} hugs_locality_probe.py '
    f'--hugs-root /content/ml-hugs '
    f'--output-dir "{HUGS_OUTPUT_DIR}" '
    f'--frame 0 --joint left_wrist --axis z --degrees 10 '
    f'--save-dir /content/probe_results'
)
run_live(
    cmd,
    cwd="/content/interactive-digital-humans/experiments/01-baseline",
    env=build_env
)


In [ ]:
import json, glob
files = sorted(glob.glob("/content/probe_results/frame*.json"))
assert files, "No probe result JSON was created."
with open(files[-1]) as f:
    result = json.load(f)
result


## 8. Save results to Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil, datetime

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
dst = Path("/content/drive/MyDrive/interactive-digital-humans/experiment-01") / stamp
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree("/content/probe_results", dst)
print("Saved to:", dst)
